In [0]:
# Databricks notebook source

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import to_date, date_format


def create_gold_dim_date(SalesOrderHeader_tbl_df):


    dim_date_df = SalesOrderHeader_tbl_df.withColumn("date_id",
                             date_format(to_date("OrderDate", "yyyy-MM-dd"), "yyyyMMdd")).withColumn("processed_timestamp", F.current_timestamp())
    dim_date_df = dim_date_df.select(
      F.col("date_id"),
      F.col("OrderDate"),
      F.col("DueDate"),
      F.col("ShipDate"),
      F.col("processed_timestamp")
    ).distinct()
                                 
    return dim_date_df



if __name__ == "__main__":

    SalesOrderHeader_tbl = dbutils.widgets.get("SalesOrderHeader")
    SalesOrderHeader_tbl_df = df = spark.read.table(SalesOrderHeader_tbl)
    dim_date_tgt = create_gold_dim_date(SalesOrderHeader_tbl_df)
    dim_date_gold_tbl = dbutils.widgets.get("dim_date_gold_tbl")
    dim_date_tgt.write.mode("overwrite").format("delta").partitionBy("OrderDate").saveAsTable(dim_date_gold_tbl)